# 第1章：降息是否一定提高企业价值？
本Notebook只使用原创假设数据。估值时点Y0年末；现金流发生于Y1—Y3年末；金额人民币亿元、股数亿股、每股价值元。三年后清算，回收40亿元已含营运资本；没有持续经营终值。先完成练习CH01-E04，再运行。

In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
if not (root / 'labs').is_dir() and (root.parent / 'labs').is_dir():
    root = root.parent
if not (root / 'labs').is_dir():
    raise RuntimeError('请从仓库根目录或notebooks目录运行')
sys.path.insert(0, str(root))
from labs.ch01_scenarios import load_case, evaluate
case = load_case()
print(case['meta'])
print(case['common'])

## 从经营到现金流
逐年FCFF = 收入×EBIT利润率×(1−经营税率)＋折旧−资本开支−经营营运资本增加。EBIT已含折旧；NWC是期末存量。检查基准第一年是否为7.25亿元。

In [ ]:
results = {name: evaluate(case, name) for name in case['scenarios']}
for name, result in results.items():
    print(name)
    for row in result['years']:
        print({key: round(value, 6) for key, value in row.items()})
    print('经营资产价值、权益价值、每股价值:',
          *(round(result[key], 6) for key in ['operating_value', 'equity_value', 'value_per_share']))
assert abs(results['base']['years'][0]['fcff'] - 7.25) < 1e-10

## 将折现率与经营变化分开
先保持基准现金流只把k从10%改为9%，再在9%下改成疲弱经营。这里是模型内条件比较，不是政策因果识别。尝试反转归因顺序，观察各项贡献是否改变。

In [ ]:
base = results['base']['operating_value']
rate_only = evaluate(case, 'base', 0.09)['operating_value']
weak = results['weakness']['operating_value']
weak_at_old_rate = evaluate(case, 'weakness', 0.10)['operating_value']
print('先利率后经营:', rate_only - base, weak - rate_only, weak - base)
print('先经营后利率:', weak_at_old_rate - base, weak - weak_at_old_rate, weak - base)
assert rate_only > base
assert weak < base
assert abs((rate_only - base) + (weak - rate_only) - (weak - base)) < 1e-10

## 利率敏感性
固定经营与回收值，不改变币种和现金流时点。思考：这张表能否说明真实降息一定提高价值？

In [ ]:
rates = [0.08, 0.09, 0.10, 0.11, 0.12]
values = [evaluate(case, 'base', rate)['operating_value'] for rate in rates]
for rate, value in zip(rates, values):
    print(f'k={rate:.0%}, V={value:.6f} 亿元')
assert all(left > right for left, right in zip(values, values[1:]))

## 读者实验
把需求改善情景中的营运资本比例由20%提高至25%，Y0营运资本仍为18；其他输入及总清算回收40保持不变。运行前预测方向。注意：保持总回收不变是额外假设，不能推断真实坏账损失。

In [ ]:
from copy import deepcopy
modified = deepcopy(case)
for row in modified['scenarios']['recovery']['years']:
    row['operating_nwc'] = 0.25 * row['revenue']
modified_result = evaluate(modified, 'recovery')
original = results['recovery']['operating_value']
print('NWC 25%时价值:', modified_result['operating_value'])
print('与NWC 20%相比变化:', modified_result['operating_value'] - original)
assert modified_result['operating_value'] < original
assert case['scenarios']['recovery']['years'][0]['operating_nwc'] == 20.8

## v0.3：交互项与对称归因
先用四角点手算交互项，再计算两个顺序的平均贡献。此归因约定不识别政策因果。

In [ ]:
from math import isclose
from labs.ch01_scenarios import symmetric_attribution
attribution = symmetric_attribution(case)
print(attribution)
assert isclose(attribution['rate'] + attribution['operations'], attribution['total'])
assert isclose(symmetric_attribution(case, new_rate=.10)['rate'], 0, abs_tol=1e-12)
assert isclose(symmetric_attribution(case, new_scenario='base')['operations'], 0, abs_tol=1e-12)

## 退出任务
用150字回答：模型支持什么条件命题，不能支持什么现实结论？补充一个真实检验需要的数据字段与时间约束。不得将这些模拟输出写成历史回测收益。